# GeoVision — YOLOv8 detector training (Kaggle)

Runs `ai/training/train_detector.py` (Module 08) on a Kaggle GPU.

> **This cannot produce a real result yet.** As of 2026-08-27, `dataset/labels/` has no
> bounding-box annotations — annotation has not started (it is the slowest, least automatable
> item on the team's plan). This notebook is written and ready so the moment annotated data
> exists, training is a matter of uploading it and pressing Run, not writing new code under
> time pressure.

**Before running, upload two Kaggle Datasets and attach them** (sidebar "Add Input"):

1. **`geovision-ai-src`** — same as the classifier notebook, a zip of this repo's `ai/` folder.
2. **`geovision-detection-data`** — the annotated YOLO-format dataset once it exists: images +
   label `.txt` files laid out the way `dataset/labels/detection/data.yaml` describes, plus that
   `data.yaml` itself (update its `path`/`train`/`val`/`test` entries to match whatever layout
   the annotation export actually produces — the checked-in one is a placeholder).

**Set the accelerator**: Settings → Accelerator → GPU. YOLOv8 on CPU is, in the training
recipe's own words, "painfully slow" — this module genuinely needs the GPU, more than the
classifier does.


In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))
else:
    print("No GPU attached. YOLOv8 training will be extremely slow — fix this before Run All.")


In [ ]:
# --- Install the ai package (--no-deps, same reasoning as the classifier notebook) plus ultralytics ---
!cp -r /kaggle/input/geovision-ai-src/ai /kaggle/working/ai
!pip install -e /kaggle/working/ai --no-deps -q
!pip install -q "ultralytics>=8.3,<9"


In [ ]:
# --- Configuration: edit to match your attached dataset's data.yaml ---
from pathlib import Path

DATA_YAML = Path("/kaggle/input/geovision-detection-data/data.yaml")
RUN_PROJECT = Path("/kaggle/working/outputs/runs/detector")

assert DATA_YAML.is_file(), f"{DATA_YAML} not found — check the dataset slug in Add Input"
print(DATA_YAML.read_text())


In [ ]:
# --- Train (Module-08-YOLO-Detection.md's recipe) ---
from ai.training.train_detector import main as train_detector_main

train_detector_main([
    "--data", str(DATA_YAML),
    "--model", "yolov8n.pt",
    "--epochs", "100",
    "--imgsz", "640",
    "--batch", "16",
    "--patience", "20",
    "--device", "0",
    "--project", str(RUN_PROJECT),
])


## Getting the checkpoint out

Same as the classifier notebook: `RUN_PROJECT` sits under `/kaggle/working/`, so Kaggle's
**Output** tab has `weights/best.pt` plus ultralytics' own training curves and PR curves
directly after the run finishes. Publish `best.pt` the same way — a Kaggle Dataset/Model for the
working copy, a GitHub Release for the one that goes in front of the panel (Open-Questions Q10).
